[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus_optimization/03_gradient_descent_mechanics/exercises.ipynb)

# Exercises — Topic 03: Gradient Descent Mechanics

20 fully solved problems in four levels: **Level 0** Concept Check (4), **Level 1** Foundation (6), **Level 2** Applications in AI/ML (6), **Level 3** Challenge (4).

## Level 0 — Concept Check

### Problem L0.1: Reading the update rule

Write down the gradient descent update and state precisely what each symbol contributes. Then explain, without computation, why the *sign* in front of $\eta$ must be negative.

**Solution.**

The update is

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\,\nabla f(\mathbf{x}_k).
$$

- $\mathbf{x}_k \in \mathbb{R}^d$ — the current parameter vector (all weights of a model, flattened).
- $\nabla f(\mathbf{x}_k) \in \mathbb{R}^d$ — the gradient: the vector whose direction is that of *steepest increase* of $f$ at $\mathbf{x}_k$, and whose length is the maximal directional derivative there.
- $\eta \gt 0$ — the step size, converting a gradient (units of $f$ per unit of $\mathbf{x}$) into a displacement (units of $\mathbf{x}$).

*Why the minus sign.* The directional derivative of $f$ at $\mathbf{x}_k$ along a unit vector $\mathbf{u}$ is $\nabla f(\mathbf{x}_k)^\top\mathbf{u}$, and by Cauchy–Schwarz this is minimized over $\lVert \mathbf{u} \rVert_2 = 1$ at $\mathbf{u} = -\nabla f(\mathbf{x}_k)/\lVert \nabla f(\mathbf{x}_k) \rVert_2$, giving the value $-\lVert \nabla f(\mathbf{x}_k) \rVert_2$. So the negative gradient is the unique steepest-*descent* direction in the Euclidean norm.

$$
\boxed{\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\nabla f(\mathbf{x}_k), \qquad \arg\min_{\lVert \mathbf{u} \rVert_2 = 1} \nabla f^\top \mathbf{u} = -\frac{\nabla f}{\lVert \nabla f \rVert_2}}
$$

*Key takeaway:* "Steepest" is norm-dependent. Change the norm defining the unit ball and the steepest direction changes — which is exactly what preconditioners, natural gradient, and Adam do.

### Problem L0.2: The stability window in one dimension

For $f(x) = \frac{1}{2}\lambda x^2$ with $\lambda = 4$, which constant step sizes $\eta$ make gradient descent converge to $0$? For which does it converge *monotonically*, and for which does it oscillate?

**Solution.**

Here $f'(x) = \lambda x = 4x$, so the update is $x_{k+1} = x_k - 4\eta x_k = (1-4\eta)x_k$ and $x_k = (1-4\eta)^k x_0$.

Convergence for every $x_0$ requires $\lvert 1-4\eta \rvert \lt 1$, i.e. $-1 \lt 1-4\eta \lt 1$, i.e.

$$
0 \lt \eta \lt \frac{2}{4} = 0.5 .
$$

Within that window the sign of $r = 1-4\eta$ decides the shape: $r \gt 0$ (i.e. $\eta \lt 0.25$) gives monotone decay; $r = 0$ ($\eta = 0.25 = 1/\lambda$) converges in a single step; $r \lt 0$ ($0.25 \lt \eta \lt 0.5$) gives sign-alternating decay — visible oscillation that is nevertheless converging. At $\eta = 0.5$ the iterate cycles $x_0 \to -x_0 \to x_0$ forever; beyond it, divergence.

$$
\boxed{\text{converges iff } 0 \lt \eta \lt 0.5; \ \text{monotone for } \eta \lt 0.25; \ \text{one step at } \eta = 0.25; \ \text{oscillatory for } 0.25 \lt \eta \lt 0.5}
$$

*Key takeaway:* Oscillation is not divergence. The failure threshold is $\eta = 2/\lambda$, while the "too large but still fine" regime begins already at $1/\lambda$.

### Problem L0.3: Sublinear versus linear rates

A convex smooth problem gives $f(\mathbf{x}_k) - f^\star \le C/k$; a strongly convex one gives $f(\mathbf{x}_k)-f^\star \le C\rho^k$ with $\rho = 0.9$. In each case, how many additional iterations are needed to gain one extra decimal digit of accuracy?

**Solution.**

**Sublinear $O(1/k)$.** To go from error $\epsilon$ to $\epsilon/10$ we need $C/k' = \epsilon/10$ where $C/k = \epsilon$, hence $k' = 10k$: the *total* iteration count must be multiplied by ten. The extra work for one more digit is $9k$ — it grows without bound as training proceeds.

**Linear (geometric) $\rho^k$.** We need $\rho^{\Delta k} = 1/10$, i.e.

$$
\Delta k = \frac{\log 10}{\log(1/\rho)} = \frac{2.3026}{\log(1/0.9)} = \frac{2.3026}{0.10536} \approx 21.9 \approx 22 \text{ iterations},
$$

*independent of how far you already are*. Each fixed block of $\approx 22$ steps buys one more digit forever.

$$
\boxed{O(1/k): \ k \mapsto 10k \ \text{per digit}; \qquad \rho^k: \ \Delta k = \log 10/\log(1/\rho) \approx 22 \ \text{per digit}}
$$

*Key takeaway:* "Sublinear" and "linear" describe qualitatively different experiences. This is why $\ell_2$ regularization (which manufactures strong convexity) can transform an optimization problem, not merely regularize it.

### Problem L0.4: What the condition number predicts

A quadratic loss has Hessian $H = \operatorname{diag}(1, 100)$. Compute $\lambda_{\max}$, $\lambda_{\min}$, $\kappa$, the stability ceiling, the optimal constant step, and the resulting per-step contraction factor.

**Solution.**

Directly from the diagonal: $\lambda_{\min} = 1$, $\lambda_{\max} = 100$, so

$$
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}} = 100 .
$$

Stability ceiling: $\eta \lt 2/\lambda_{\max} = 0.02$. Optimal constant step:

$$
\eta^\star = \frac{2}{\lambda_{\min}+\lambda_{\max}} = \frac{2}{101} \approx 0.0198 .
$$

Contraction factor at $\eta^\star$:

$$
\rho^\star = \frac{\kappa-1}{\kappa+1} = \frac{99}{101} \approx 0.980 .
$$

Only $2\%$ of the error is removed per step, so about $\log(1/\epsilon)/\log(1/0.980) \approx 50\log(1/\epsilon)$ iterations are needed — roughly $115$ steps per digit.

$$
\boxed{\kappa = 100, \ \eta \lt 0.02, \ \eta^\star = 2/101, \ \rho^\star = 99/101 \approx 0.980}
$$

*Key takeaway:* The optimal step sits just below the stability ceiling, and even then progress is limited by $\lambda_{\min}$. A ratio of 100 between curvatures — mild by deep-learning standards — already costs a hundredfold slowdown relative to a well-conditioned problem.

## Level 1 — Foundation

### Problem L1.1: Closed-form iterates on a quadratic

For $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top H\mathbf{x} - \mathbf{b}^\top\mathbf{x}$ with $H \succ 0$, derive a closed-form expression for $\mathbf{x}_k$ in terms of $\mathbf{x}_0$, and state exactly when $\mathbf{x}_k \to \mathbf{x}^\star$ for every starting point.

**Solution.**

**Step 1.** $\nabla f(\mathbf{x}) = H\mathbf{x}-\mathbf{b}$; the unique stationary point solves $H\mathbf{x}^\star = \mathbf{b}$, so $\mathbf{x}^\star = H^{-1}\mathbf{b}$ (and it is the global minimizer since $H \succ 0$).

**Step 2.** Because $\mathbf{b} = H\mathbf{x}^\star$, the gradient is a linear function of the error $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^\star$: $\nabla f(\mathbf{x}_k) = H(\mathbf{x}_k - \mathbf{x}^\star) = H\mathbf{e}_k$. Subtracting $\mathbf{x}^\star$ from the update,

$$
\mathbf{e}_{k+1} = \mathbf{e}_k - \eta H\mathbf{e}_k = (I-\eta H)\mathbf{e}_k \quad \Longrightarrow \quad \mathbf{e}_k = (I-\eta H)^k\mathbf{e}_0 .
$$

Hence $\mathbf{x}_k = \mathbf{x}^\star + (I-\eta H)^k(\mathbf{x}_0 - \mathbf{x}^\star)$.

**Step 3.** Write $H = Q\Lambda Q^\top$. Then $(I-\eta H)^k = Q(I-\eta\Lambda)^k Q^\top$, and in the rotated coordinates $\tilde{e}_{k,i} = (1-\eta\lambda_i)^k\tilde{e}_{0,i}$. This tends to zero for *all* $\mathbf{e}_0$ iff $\lvert 1-\eta\lambda_i \rvert \lt 1$ for every $i$, i.e. iff $0 \lt \eta \lt 2/\lambda_{\max}$.

$$
\boxed{\mathbf{x}_k = \mathbf{x}^\star + (I-\eta H)^k(\mathbf{x}_0-\mathbf{x}^\star); \quad \text{convergence} \iff 0 \lt \eta \lt 2/\lambda_{\max}}
$$

*Key takeaway:* On quadratics gradient descent is a linear recursion whose entire behavior is the spectrum of $I-\eta H$. Every qualitative statement about training curves is a statement about the numbers $1-\eta\lambda_i$.

### Problem L1.2: A full numerical audit of a 2D quadratic

Let $H = \operatorname{diag}(1, 10)$ and $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top H\mathbf{x}$. Compute (a) the stability ceiling, (b) the step $\eta = 1/L$ and its two mode factors, (c) the optimal step and its contraction, (d) the number of iterations to reduce the error by $10^{-3}$ at each of those steps.

**Solution.**

**(a)** $\lambda_{\max} = L = 10$, so stability requires $\eta \lt 2/10 = 0.2$.

**(b)** At $\eta = 1/L = 0.1$: mode factors are $1-0.1\cdot 1 = 0.9$ and $1-0.1\cdot 10 = 0$. The steep direction is annihilated in one step; the flat one contracts by $0.9$, so $\rho_{1/L} = 0.9$.

**(c)** $\eta^\star = \dfrac{2}{1+10} = \dfrac{2}{11} \approx 0.1818$. Mode factors: $1 - \frac{2}{11} = \frac{9}{11} \approx 0.818$ and $1 - \frac{20}{11} = -\frac{9}{11} \approx -0.818$; equal magnitudes, opposite signs, so

$$
\rho^\star = \frac{\kappa-1}{\kappa+1} = \frac{9}{11} \approx 0.818, \qquad \kappa = 10 .
$$

**(d)** Iterations for $\rho^k \le 10^{-3}$, i.e. $k \ge \dfrac{3\log 10}{\log(1/\rho)}$:

| Step | $\rho$ | $\log(1/\rho)$ | $k$ |
|---|---|---|---|
| $\eta = 1/L = 0.1$ | $0.900$ | $0.1054$ | $\lceil 6.908/0.1054 \rceil = 66$ |
| $\eta^\star = 2/11$ | $0.818$ | $0.2007$ | $\lceil 6.908/0.2007 \rceil = 35$ |

$$
\boxed{\eta \lt 0.2; \quad \rho_{1/L} = 0.9 \ (66 \text{ steps}); \quad \eta^\star = 2/11,\ \rho^\star = 9/11 \ (35 \text{ steps})}
$$

*Key takeaway:* Even the "safe, principled" step $1/L$ is roughly twice as slow as the optimal one here. The gap widens with $\kappa$: $1/L$ gives $\rho = 1-1/\kappa$ while $\eta^\star$ gives $1-2/(\kappa+1)$ — a factor of two in iteration count, which is why $\eta$ is tuned upward toward the ceiling in practice.

### Problem L1.3: Sufficient decrease and the best guaranteed step

Let $f$ be $L$-smooth. Starting from the descent lemma, derive the guaranteed one-step decrease as a function of $\eta$, find the range of $\eta$ for which progress is guaranteed, and find the $\eta$ maximizing the guarantee.

**Solution.**

Apply the descent lemma with $\mathbf{y} = \mathbf{x}_{k+1} = \mathbf{x}_k - \eta\mathbf{g}_k$, $\mathbf{g}_k = \nabla f(\mathbf{x}_k)$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \eta\lVert \mathbf{g}_k \rVert_2^2 + \frac{L\eta^2}{2}\lVert \mathbf{g}_k \rVert_2^2 = f(\mathbf{x}_k) - c(\eta)\lVert \mathbf{g}_k \rVert_2^2,
$$

where the *guaranteed-decrease coefficient* is

$$
c(\eta) = \eta - \frac{L\eta^2}{2} = \eta\left(1-\frac{L\eta}{2}\right).
$$

**Range.** $c(\eta) \gt 0$ exactly when $0 \lt \eta \lt 2/L$ — the smoothness-based stability window, matching $\eta \lt 2/\lambda_{\max}$ on quadratics since there $L = \lambda_{\max}$.

**Optimum.** $c$ is a downward parabola in $\eta$ with $c'(\eta) = 1 - L\eta$, so the maximum is at $\eta = 1/L$ with $c(1/L) = \frac{1}{L}-\frac{1}{2L} = \frac{1}{2L}$:

$$
f\!\left(\mathbf{x}_k - \tfrac{1}{L}\mathbf{g}_k\right) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \mathbf{g}_k \rVert_2^2 .
$$

Equivalently, $\eta = 1/L$ is the exact minimizer of the certified quadratic upper model $f(\mathbf{x}_k)+\mathbf{g}_k^\top\mathbf{h}+\frac{L}{2}\lVert \mathbf{h} \rVert_2^2$ along the direction $-\mathbf{g}_k$.

$$
\boxed{c(\eta) = \eta\left(1-\tfrac{L\eta}{2}\right) \gt 0 \iff 0 \lt \eta \lt \tfrac{2}{L}; \quad \max_\eta c = \tfrac{1}{2L} \text{ at } \eta = \tfrac{1}{L}}
$$

*Key takeaway:* $\eta = 1/L$ is not a heuristic — it is the argmin of a *provable* upper bound on the loss. The gap between $1/L$ (best guarantee) and $2/(\mu+L)$ (best actual rate) is the gap between worst-case pessimism and exact analysis.

### Problem L1.4: A convergence guarantee with no convexity

Let $f$ be $L$-smooth and bounded below by $f^\star$, and run gradient descent with $\eta = 1/L$. Prove that after $k$ steps some visited point has a small gradient:

$$
\min_{0 \le t \lt k}\lVert \nabla f(\mathbf{x}_t) \rVert_2 \le \sqrt{\frac{2L\left(f(\mathbf{x}_0)-f^\star\right)}{k}} .
$$

**Solution.**

**Step 1 (per-step budget).** By Problem L1.3 with $\eta = 1/L$,

$$
f(\mathbf{x}_{t+1}) \le f(\mathbf{x}_t) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \quad \Longrightarrow \quad \frac{1}{2L}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le f(\mathbf{x}_t) - f(\mathbf{x}_{t+1}) .
$$

**Step 2 (telescope).** Sum for $t=0,\dots,k-1$. The right side collapses:

$$
\frac{1}{2L}\sum_{t=0}^{k-1}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le f(\mathbf{x}_0) - f(\mathbf{x}_k) \le f(\mathbf{x}_0) - f^\star ,
$$

using only that $f$ is bounded below.

**Step 3 (min $\le$ average).**

$$
k\min_{0 \le t \lt k}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le \sum_{t=0}^{k-1}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le 2L\left(f(\mathbf{x}_0)-f^\star\right),
$$

and taking square roots gives the claim. $\blacksquare$

$$
\boxed{\min_{0 \le t \lt k}\lVert \nabla f(\mathbf{x}_t) \rVert_2 \le \sqrt{\frac{2L\left(f(\mathbf{x}_0)-f^\star\right)}{k}} = O\!\left(k^{-1/2}\right)}
$$

*Key takeaway:* This is the strongest guarantee available for a generic deep network: gradient descent reaches an $\epsilon$-stationary point in $O(1/\epsilon^2)$ steps. It promises *stationarity*, never *optimality* — which is exactly why the landscape questions of Topic 04 matter.

### Problem L1.5: Strong convexity gives a geometric rate

Show that $\mu$-strong convexity implies the Polyak–Łojasiewicz inequality $\lVert \nabla f(\mathbf{x}) \rVert_2^2 \ge 2\mu\left(f(\mathbf{x})-f^\star\right)$, and deduce that gradient descent with $\eta = 1/L$ satisfies $\delta_k \le (1-1/\kappa)^k\delta_0$.

**Solution.**

**Step 1 (PL from strong convexity).** For all $\mathbf{y}$,

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) + \frac{\mu}{2}\lVert \mathbf{y}-\mathbf{x} \rVert_2^2 =: q(\mathbf{y}).
$$

The right side is a strictly convex quadratic in $\mathbf{y}$; setting $\nabla q = \nabla f(\mathbf{x}) + \mu(\mathbf{y}-\mathbf{x}) = 0$ gives $\mathbf{y}_{\min} = \mathbf{x}-\frac{1}{\mu}\nabla f(\mathbf{x})$ and

$$
\min_{\mathbf{y}} q(\mathbf{y}) = f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x}) \rVert_2^2 .
$$

Since $f \ge q$ pointwise, $f^\star = \min_\mathbf{y} f(\mathbf{y}) \ge \min_\mathbf{y} q(\mathbf{y})$, which rearranges to

$$
\lVert \nabla f(\mathbf{x}) \rVert_2^2 \ge 2\mu\left(f(\mathbf{x})-f^\star\right).
$$

**Step 2 (contraction).** Combine with sufficient decrease at $\eta = 1/L$ (Problem L1.3), writing $\delta_t = f(\mathbf{x}_t)-f^\star$:

$$
\delta_{t+1} \le \delta_t - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le \delta_t - \frac{2\mu}{2L}\delta_t = \left(1-\frac{\mu}{L}\right)\delta_t .
$$

**Step 3 (iterate and convert to a complexity).** $\delta_k \le (1-1/\kappa)^k\delta_0 \le e^{-k/\kappa}\delta_0$, so $\delta_k \le \epsilon$ once $k \ge \kappa\log(\delta_0/\epsilon)$.

$$
\boxed{\delta_k \le \left(1-\frac{1}{\kappa}\right)^k\delta_0, \qquad k_\epsilon = O\!\left(\kappa\log\frac{1}{\epsilon}\right), \quad \kappa = \frac{L}{\mu}}
$$

*Key takeaway:* PL is the working hypothesis, not convexity itself: any function satisfying "small gradient implies near-optimal" converges geometrically, and some nonconvex losses (e.g. wide overparameterized networks near their interpolating manifold) do satisfy PL locally.

### Problem L1.6: Why the zig-zag happens — orthogonal successive steps

Show that gradient descent with *exact line search* produces successive search directions that are orthogonal: $\nabla f(\mathbf{x}_{k+1})^\top \nabla f(\mathbf{x}_k) = 0$. Then explain the zig-zag on an elongated quadratic valley.

**Solution.**

**Step 1 (the line-search optimality condition).** Exact line search chooses $\eta_k = \arg\min_{\eta \gt 0}\phi(\eta)$ where $\phi(\eta) = f(\mathbf{x}_k - \eta\nabla f(\mathbf{x}_k))$. At an interior minimum $\phi'(\eta_k) = 0$. By the chain rule,

$$
\phi'(\eta) = \nabla f\!\left(\mathbf{x}_k - \eta\nabla f(\mathbf{x}_k)\right)^\top\left(-\nabla f(\mathbf{x}_k)\right).
$$

Setting $\eta = \eta_k$ and using $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta_k\nabla f(\mathbf{x}_k)$:

$$
\nabla f(\mathbf{x}_{k+1})^\top\nabla f(\mathbf{x}_k) = 0 .
$$

**Step 2 (geometric meaning).** Each step travels until the loss stops decreasing along the current line, which happens precisely when the new gradient is perpendicular to the direction just travelled. Consecutive steps therefore turn by exactly $90^\circ$.

**Step 3 (why that is bad in a valley).** On $f = \frac{1}{2}(\lambda_1 x_1^2 + \lambda_2 x_2^2)$ with $\lambda_2 \gg \lambda_1$, the level sets are ellipses of axis ratio $\sqrt{\kappa}$. The gradient at a generic point is dominated by the steep coordinate and points almost *across* the valley rather than along it, so the right-angle sequence produces a staircase that crosses the floor repeatedly while creeping along it. One can show the error contracts by exactly $\left(\frac{\kappa-1}{\kappa+1}\right)$ per step in the worst case (Kantorovich), so exact line search does not beat the optimal constant step.

$$
\boxed{\nabla f(\mathbf{x}_{k+1})^\top\nabla f(\mathbf{x}_k) = 0 \implies \text{right-angle zig-zag; rate still } \tfrac{\kappa-1}{\kappa+1}}
$$

*Key takeaway:* Zig-zagging is not caused by a badly tuned step size — even the *perfect* step size zig-zags. The cure must change the *direction* (momentum, conjugate gradients, preconditioning), not the length.

## Level 2 — Applications in AI/ML

### Problem L2.1: Setting the learning rate for linear regression from data

For $f(\mathbf{w}) = \frac{1}{n}\lVert X\mathbf{w}-\mathbf{y} \rVert_2^2$ with $n = 500$ and singular values of $X$ equal to $\sigma_1 = 50$, $\sigma_2 = 10$, $\sigma_3 = 2$, compute $L$, $\mu$, $\kappa$, the stability ceiling, and the optimal step. Then repeat with ridge regularization $+\lambda\lVert \mathbf{w} \rVert_2^2$, $\lambda = 1$.

**Solution.**

**Unregularized.** $\nabla f(\mathbf{w}) = \frac{2}{n}X^\top(X\mathbf{w}-\mathbf{y})$ and $\nabla^2 f = \frac{2}{n}X^\top X$, a constant matrix with eigenvalues $\frac{2}{n}\sigma_i^2$:

$$
L = \frac{2}{500}\cdot 2500 = 10, \qquad \mu = \frac{2}{500}\cdot 4 = 0.016, \qquad \kappa = \frac{L}{\mu} = \frac{\sigma_1^2}{\sigma_3^2} = 625 .
$$

Stability: $\eta \lt 2/L = 0.2$. Optimal: $\eta^\star = \frac{2}{\mu+L} = \frac{2}{10.016} \approx 0.1997$, with $\rho^\star = \frac{624}{626} \approx 0.9968$ — about $720$ iterations per digit.

**With ridge $\lambda = 1$.** The Hessian becomes $\frac{2}{n}X^\top X + 2\lambda I$, shifting every eigenvalue by $2\lambda = 2$:

$$
L' = 12, \qquad \mu' = 2.016, \qquad \kappa' = \frac{12}{2.016} \approx 5.95 .
$$

Now $\eta \lt 2/12 \approx 0.167$, $\eta^\star = \frac{2}{14.016} \approx 0.1427$, and $\rho^\star = \frac{\kappa'-1}{\kappa'+1} \approx 0.712$ — about $6.8$ iterations per digit, a **hundredfold** speedup.

$$
\boxed{\kappa = 625 \to \kappa' \approx 5.95; \quad \rho^\star: 0.9968 \to 0.712}
$$

*Key takeaway:* Weight decay is a preconditioner in disguise. It shifts the whole spectrum up by $2\lambda$, which barely moves $L$ but massively lifts $\mu$ — the statistical story (shrinkage) and the optimization story (conditioning) are the same shift.

### Problem L2.2: A free learning-rate certificate for logistic regression

For $f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n \log\!\left(1+e^{-y_i \mathbf{x}_i^\top\mathbf{w}}\right)$ with $y_i \in \{-1,+1\}$, prove that $f$ is $L$-smooth with $L = \frac{1}{4n}\lambda_{\max}(X^\top X)$, and give a provably stable constant step size.

**Solution.**

**Step 1 (Hessian).** Write $s_i = \sigma(-y_i\mathbf{x}_i^\top\mathbf{w})$ with $\sigma(u) = 1/(1+e^{-u})$. Differentiating twice (Topic 01),

$$
\nabla^2 f(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^n s_i(1-s_i)\,\mathbf{x}_i\mathbf{x}_i^\top = \frac{1}{n}X^\top S X, \qquad S = \operatorname{diag}\!\left(s_i(1-s_i)\right).
$$

**Step 2 (uniform bound).** The scalar map $s \mapsto s(1-s)$ on $[0,1]$ is maximized at $s = \frac{1}{2}$, where it equals $\frac{1}{4}$. Hence $0 \preceq S \preceq \frac{1}{4}I$, and since congruence preserves the ordering,

$$
0 \preceq \nabla^2 f(\mathbf{w}) \preceq \frac{1}{4n}X^\top X \quad \text{for every } \mathbf{w}.
$$

**Step 3 (smoothness constant).** Therefore $\lambda_{\max}(\nabla^2 f) \le \frac{1}{4n}\lambda_{\max}(X^\top X) = \frac{\sigma_{\max}(X)^2}{4n} =: L$, and a uniform Hessian bound is exactly $L$-smoothness.

**Step 4 (step size).** Stability needs $\eta \lt 2/L = \frac{8n}{\sigma_{\max}(X)^2}$, and the certified choice is $\eta = 1/L = \frac{4n}{\sigma_{\max}(X)^2}$. Concretely, for standardized features with $\lVert \mathbf{x}_i \rVert_2^2 \le R^2$ one has $\lambda_{\max}(X^\top X) \le nR^2$, giving the data-free bound $L \le R^2/4$ and $\eta = 4/R^2$.

$$
\boxed{L = \frac{\lambda_{\max}(X^\top X)}{4n}, \qquad \eta_{\text{safe}} \lt \frac{8n}{\sigma_{\max}(X)^2}, \qquad \eta_{1/L} = \frac{4n}{\sigma_{\max}(X)^2}}
$$

*Key takeaway:* For generalized linear models the learning rate is a computation, not a hyperparameter search. Note also $\mu = 0$ here (the Hessian is singular for separable data), so unregularized logistic regression is convex but *not* strongly convex — the rate is $O(1/k)$ until you add weight decay.

### Problem L2.3: Minibatch noise, batch size, and the loss floor

Per-example gradients have variance bound $\sigma^2 = 4$; the problem is $L$-smooth with $L = 10$ and $\mu$-strongly convex with $\mu = 0.5$. Using the SGD bound $\mathbb{E}[\delta_k] \le (1-\eta\mu)^k\delta_0 + \frac{\eta L\sigma^2}{2\mu B}$, find the asymptotic loss floor for $(\eta, B) = (0.05, 32)$, and two distinct ways to cut it by a factor of $4$.

**Solution.**

**Floor.** As $k \to \infty$ the transient $(1-\eta\mu)^k \to 0$ (since $\eta\mu = 0.025 \in (0,1)$), leaving

$$
\delta_\infty \le \frac{\eta L\sigma^2}{2\mu B} = \frac{0.05 \times 10 \times 4}{2 \times 0.5 \times 32} = \frac{2}{32} = 0.0625 .
$$

**Two ways to divide it by 4.** The bound depends on $\eta$ and $B$ only through the ratio $\eta/B$, so:

1. **Decay the step:** $\eta \to \eta/4 = 0.0125$ with $B = 32$ gives $\delta_\infty \le 0.015625$. Cost: the contraction slows from $(1-0.025)$ to $(1-0.00625)$ — four times more iterations per digit.
2. **Grow the batch:** $B \to 4B = 128$ with $\eta = 0.05$ gives the same floor, and *keeps* the fast contraction rate — but each iteration costs four times more gradient evaluations.

$$
\boxed{\delta_\infty \le \frac{\eta L\sigma^2}{2\mu B} = 0.0625; \quad \text{halve it via } \eta/B \text{ — decay } \eta \text{ or grow } B}
$$

*Key takeaway:* Constant-step SGD does not converge to the minimizer; it converges to a *noise ball* of radius $\propto \eta/B$. The equivalence of $\eta$-decay and $B$-growth (Smith et al., 2018), and the linear scaling rule $\eta \propto B$ (Goyal et al., 2017), are both immediate readings of this one ratio — valid only while $\eta$ stays under the ceiling $2/L = 0.2$.

### Problem L2.4: Which schedules are provably valid

Check the Robbins–Monro conditions $\sum_k \eta_k = \infty$ and $\sum_k \eta_k^2 \lt \infty$ for $\eta_k = c/k$, $\eta_k = c/\sqrt{k}$, and $\eta_k \equiv c$. Interpret each in training terms.

**Solution.**

The two conditions have complementary jobs: $\sum\eta_k = \infty$ means the iterates retain enough total "travel" to reach the optimum from any start; $\sum\eta_k^2 \lt \infty$ means the accumulated *noise* injected by stochastic gradients is finite, so the noise ball shrinks to a point.

| Schedule | $\sum_k \eta_k$ | $\sum_k \eta_k^2$ | Verdict |
|---|---|---|---|
| $\eta_k = c/k$ | $c\sum 1/k$ diverges (harmonic) | $c^2\sum 1/k^2 = c^2\pi^2/6 \lt \infty$ | **Both hold** — provably convergent |
| $\eta_k = c/\sqrt{k}$ | $c\sum k^{-1/2}$ diverges | $c^2\sum 1/k$ diverges | Travel is fine, noise is not summable — converges only in an averaged/$O(1/\sqrt{k})$ sense |
| $\eta_k \equiv c$ | diverges | $\sum c^2$ diverges | Noise ball of radius $\propto c$ never closes |

The general rule for $\eta_k = ck^{-p}$: the first condition needs $p \le 1$, the second needs $p \gt 1/2$, so the admissible window is $p \in (1/2, 1]$.

$$
\boxed{\eta_k = ck^{-p} \text{ satisfies Robbins–Monro} \iff \tfrac{1}{2} \lt p \le 1}
$$

*Key takeaway:* The classic $1/k$ schedule is the theoretically canonical one, yet practice prefers step decay, cosine, and warmup — because asymptotic conditions say nothing about the first few thousand steps, where staying inside the trust radius of the local model (Topic 02) matters more than the tail behavior.

### Problem L2.5: What momentum buys on an ill-conditioned problem

A quadratic has $\mu = 0.01$ and $L = 100$. Compare the iteration counts to reach $10^{-6}$ relative error for (a) optimally tuned gradient descent and (b) optimally tuned heavy-ball momentum, and give the optimal $\beta$.

**Solution.**

**Conditioning.** $\kappa = L/\mu = 100/0.01 = 10^4$, so $\sqrt{\kappa} = 100$.

**(a) Gradient descent.** $\rho_{\mathrm{GD}} = \dfrac{\kappa-1}{\kappa+1} = \dfrac{9999}{10001} \approx 0.9998$, and $\log(1/\rho_{\mathrm{GD}}) \approx \dfrac{2}{\kappa} = 2\times 10^{-4}$. To reach $10^{-6}$:

$$
k_{\mathrm{GD}} \approx \frac{\log(10^6)}{2\times 10^{-4}} = \frac{13.82}{2\times10^{-4}} \approx 6.9\times 10^{4}\ \text{iterations}.
$$

**(b) Heavy ball.** $\rho_{\mathrm{HB}} = \dfrac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1} = \dfrac{99}{101} \approx 0.9802$, with $\log(1/\rho_{\mathrm{HB}}) \approx 2/\sqrt{\kappa} = 0.02$:

$$
k_{\mathrm{HB}} \approx \frac{13.82}{0.02} \approx 6.9\times 10^{2}\ \text{iterations}.
$$

**Tuning.** $\beta^\star = \left(\dfrac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2 = (0.9802)^2 \approx 0.961$ and $\eta^\star = \dfrac{4}{\left(\sqrt{L}+\sqrt{\mu}\right)^2} = \dfrac{4}{(10+0.1)^2} \approx 0.0392$.

$$
\boxed{k_{\mathrm{GD}} \approx 6.9\times10^4 \ \text{vs}\ k_{\mathrm{HB}} \approx 6.9\times10^2 \ (100\times), \quad \beta^\star \approx 0.961}
$$

*Key takeaway:* The speedup is exactly $\sqrt{\kappa}$, and it costs one extra buffer of memory. This is also why the practical default $\beta = 0.9$ implicitly assumes $\kappa \approx 4\times10^2$: raising $\beta$ toward $0.99$ is the right move on stiffer problems, provided $\eta$ comes down with it.

### Problem L2.6: Backpropagation as the chain rule, and its cost

For the two-layer network $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x}$, $\mathbf{a}^{(1)} = \phi(\mathbf{z}^{(1)})$, $\hat{y} = \mathbf{w}^{(2)\top}\mathbf{a}^{(1)}$, with squared loss $\mathcal{L} = \frac{1}{2}(\hat{y}-y)^2$, derive $\partial\mathcal{L}/\partial W^{(1)}$ and $\partial\mathcal{L}/\partial\mathbf{w}^{(2)}$, and state the cost of one gradient relative to one forward pass.

**Solution.**

**Step 1 (output layer).** Let $r = \hat{y}-y$, so $\partial\mathcal{L}/\partial\hat{y} = r$. Since $\hat{y} = \mathbf{w}^{(2)\top}\mathbf{a}^{(1)}$,

$$
\frac{\partial\mathcal{L}}{\partial\mathbf{w}^{(2)}} = r\,\mathbf{a}^{(1)} .
$$

**Step 2 (backpropagate to the hidden pre-activations).** Also $\partial\hat{y}/\partial\mathbf{a}^{(1)} = \mathbf{w}^{(2)}$, and $\mathbf{a}^{(1)} = \phi(\mathbf{z}^{(1)})$ acts elementwise, so the adjoint is

$$
\boldsymbol{\delta}^{(1)} = \frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}} = \left(r\,\mathbf{w}^{(2)}\right) \odot \phi'\!\left(\mathbf{z}^{(1)}\right).
$$

**Step 3 (first-layer weights).** Since $\mathbf{z}^{(1)} = W^{(1)}\mathbf{x}$ is linear in $W^{(1)}$, each weight $W^{(1)}_{ij}$ affects only $z^{(1)}_i$ with $\partial z^{(1)}_i/\partial W^{(1)}_{ij} = x_j$, giving the outer product

$$
\frac{\partial\mathcal{L}}{\partial W^{(1)}} = \boldsymbol{\delta}^{(1)}\mathbf{x}^\top .
$$

**Step 4 (cost).** The backward pass reuses the stored $\mathbf{x}, \mathbf{z}^{(1)}, \mathbf{a}^{(1)}$ and performs one matrix–vector product per layer, the same asymptotic work as the forward pass. Hence the *entire* gradient with respect to all $d$ parameters costs $O(1)$ forward passes (in practice $2\times$–$3\times$), independent of $d$ — whereas finite differences or forward-mode AD would cost $O(d)$ passes.

$$
\boxed{\frac{\partial\mathcal{L}}{\partial\mathbf{w}^{(2)}} = r\mathbf{a}^{(1)}, \quad \frac{\partial\mathcal{L}}{\partial W^{(1)}} = \left[\left(r\mathbf{w}^{(2)}\right)\odot\phi'(\mathbf{z}^{(1)})\right]\mathbf{x}^\top, \quad \text{cost} = O(1)\ \text{forward passes}}
$$

*Key takeaway:* Gradient descent is viable at billions of parameters only because reverse-mode AD makes $\nabla f$ cost the same as $f$. The price is memory for the stored activations — the trade that gradient checkpointing renegotiates.

## Level 3 — Challenge

### Problem L3.1: Prove the $O(1/k)$ rate for convex $L$-smooth functions

Let $f$ be convex and $L$-smooth with a minimizer $\mathbf{x}^\star$. Prove that gradient descent with $\eta = 1/L$ satisfies

$$
f(\mathbf{x}_k) - f^\star \le \frac{L\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2}{2k} .
$$

**Solution.**

Write $\delta_t = f(\mathbf{x}_t)-f^\star$, $\mathbf{g}_t = \nabla f(\mathbf{x}_t)$, $R_t = \lVert \mathbf{x}_t-\mathbf{x}^\star \rVert_2^2$, and $\eta = 1/L$.

**Step 1 (expand the distance).**

$$
R_{t+1} = \lVert \mathbf{x}_t - \eta\mathbf{g}_t - \mathbf{x}^\star \rVert_2^2 = R_t - 2\eta\,\mathbf{g}_t^\top(\mathbf{x}_t-\mathbf{x}^\star) + \eta^2\lVert \mathbf{g}_t \rVert_2^2 .
$$

**Step 2 (convexity bounds the cross term from below).** The gradient inequality at $\mathbf{x}_t$ evaluated at $\mathbf{x}^\star$ reads $f^\star \ge f(\mathbf{x}_t) + \mathbf{g}_t^\top(\mathbf{x}^\star-\mathbf{x}_t)$, hence

$$
\mathbf{g}_t^\top(\mathbf{x}_t-\mathbf{x}^\star) \ge \delta_t .
$$

**Step 3 (smoothness bounds the quadratic term).** Sufficient decrease at $\eta = 1/L$ gives $\delta_{t+1} \le \delta_t - \frac{1}{2L}\lVert \mathbf{g}_t \rVert_2^2$, i.e. $\lVert \mathbf{g}_t \rVert_2^2 \le 2L(\delta_t-\delta_{t+1})$, so

$$
\eta^2\lVert \mathbf{g}_t \rVert_2^2 = \frac{1}{L^2}\lVert \mathbf{g}_t \rVert_2^2 \le \frac{2}{L}\left(\delta_t-\delta_{t+1}\right).
$$

**Step 4 (combine into a clean recursion).** Substituting Steps 2–3 with $\eta = 1/L$,

$$
R_{t+1} \le R_t - \frac{2}{L}\delta_t + \frac{2}{L}\left(\delta_t-\delta_{t+1}\right) = R_t - \frac{2}{L}\delta_{t+1} .
$$

The two $\delta_t$ terms cancel exactly — the crux of the proof.

**Step 5 (telescope).** Summing $t = 0,\dots,k-1$ and using $R_k \ge 0$,

$$
\frac{2}{L}\sum_{t=1}^{k}\delta_t \le R_0 - R_k \le R_0 = \lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2 .
$$

**Step 6 (use monotonicity).** Sufficient decrease makes $(\delta_t)$ non-increasing, so $k\delta_k \le \sum_{t=1}^k\delta_t$, giving

$$
\delta_k \le \frac{1}{k}\sum_{t=1}^{k}\delta_t \le \frac{L\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2}{2k}. \qquad \blacksquare
$$

$$
\boxed{f(\mathbf{x}_k)-f^\star \le \frac{L\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2}{2k}}
$$

*Key takeaway:* Only two facts were used — convexity (once, in Step 2) and smoothness (once, in Step 3) — and they enter through *different* inequalities that happen to cancel. Nesterov's method modifies the iterate sequence so that the same accounting yields $O(1/k^2)$, which his lower bound shows is optimal for gradient-only methods.

### Problem L3.2: Deriving the $\sqrt{\kappa}$ rate of heavy-ball momentum

For $f(x) = \frac{1}{2}\lambda x^2$ run heavy ball $x_{k+1} = x_k - \eta\lambda x_k + \beta(x_k - x_{k-1})$. Derive the characteristic polynomial, find the condition under which the contraction factor is $\sqrt{\beta}$ independently of $\lambda$, and deduce the optimal $\beta$ and $\eta$ for $\lambda \in [\mu, L]$.

**Solution.**

**Step 1 (linear recursion and companion matrix).** Rearranging,

$$
x_{k+1} = (1+\beta-\eta\lambda)x_k - \beta x_{k-1},
$$

a linear two-term recursion. In matrix form $\begin{bmatrix}x_{k+1}\\x_k\end{bmatrix} = T\begin{bmatrix}x_k\\x_{k-1}\end{bmatrix}$ with $T = \begin{bmatrix}1+\beta-\eta\lambda & -\beta \\ 1 & 0\end{bmatrix}$.

**Step 2 (characteristic polynomial).** $\det(T - zI) = 0$ gives

$$
z^2 - (1+\beta-\eta\lambda)z + \beta = 0 .
$$

Note the constant term: the *product* of the roots is $z_1 z_2 = \beta$, and their sum is $1+\beta-\eta\lambda$.

**Step 3 (the complex regime).** If the discriminant $(1+\beta-\eta\lambda)^2 - 4\beta$ is negative, the roots are a conjugate pair, so $z_2 = \overline{z_1}$ and $\lvert z_1 \rvert^2 = z_1\overline{z_1} = \beta$. Then

$$
\lvert z_1 \rvert = \lvert z_2 \rvert = \sqrt{\beta} \quad \text{regardless of } \lambda .
$$

This is the key structural fact: in the complex regime the decay rate is *decoupled from the curvature*, whereas plain gradient descent's rate $\lvert 1-\eta\lambda \rvert$ is entirely curvature-dependent.

**Step 4 (make it hold for the whole spectrum).** The discriminant condition is $\left\lvert 1+\beta-\eta\lambda \right\rvert \lt 2\sqrt{\beta}$, i.e.

$$
\frac{\left(1-\sqrt{\beta}\right)^2}{\eta} \lt \lambda \lt \frac{\left(1+\sqrt{\beta}\right)^2}{\eta}.
$$

To cover exactly $[\mu, L]$, take both ends tight: $\eta\mu = (1-\sqrt{\beta})^2$ and $\eta L = (1+\sqrt{\beta})^2$. Dividing,

$$
\frac{L}{\mu} = \kappa = \left(\frac{1+\sqrt{\beta}}{1-\sqrt{\beta}}\right)^2 \quad \Longrightarrow \quad \sqrt{\kappa} = \frac{1+\sqrt{\beta}}{1-\sqrt{\beta}} \quad \Longrightarrow \quad \sqrt{\beta} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}.
$$

Substituting back, $\eta = \dfrac{\left(1+\sqrt{\beta}\right)^2}{L} = \dfrac{4}{\left(\sqrt{L}+\sqrt{\mu}\right)^2}$.

**Step 5 (rate).** The contraction is $\sqrt{\beta} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1} = 1 - \frac{2}{\sqrt{\kappa}} + O(\kappa^{-1})$, so $\epsilon$-accuracy costs $O(\sqrt{\kappa}\log(1/\epsilon))$ iterations versus $O(\kappa\log(1/\epsilon))$ for gradient descent. $\blacksquare$

$$
\boxed{\beta^\star = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2, \quad \eta^\star = \frac{4}{\left(\sqrt{L}+\sqrt{\mu}\right)^2}, \quad \text{rate } = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}}
$$

*Key takeaway:* Acceleration comes from moving the dynamics into the *complex* regime, where the decay rate is set by $\sqrt{\beta}$ alone. The iterates spiral rather than zig-zag — the discrete analogue of critical damping. (Heavy ball's guarantee is quadratic-specific; Nesterov's variant achieves the same order on all smooth strongly convex functions.)

### Problem L3.3: The SGD noise ball, derived

Let $f$ be $L$-smooth and $\mu$-strongly convex, and let $g_k$ satisfy $\mathbb{E}[g_k \mid \mathbf{x}_k] = \nabla f(\mathbf{x}_k)$ and $\mathbb{E}\left[\lVert g_k - \nabla f(\mathbf{x}_k) \rVert_2^2 \mid \mathbf{x}_k\right] \le \sigma_B^2$. For constant $0 \lt \eta \le 1/L$, prove

$$
\mathbb{E}[\delta_k] \le (1-\eta\mu)^k \delta_0 + \frac{\eta L\sigma_B^2}{2\mu}.
$$

**Solution.**

**Step 1 (descent lemma with a stochastic step).** With $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta g_k$, the descent lemma gives

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \eta\,\nabla f(\mathbf{x}_k)^\top g_k + \frac{L\eta^2}{2}\lVert g_k \rVert_2^2 .
$$

**Step 2 (take conditional expectation).** Unbiasedness gives $\mathbb{E}\left[\nabla f(\mathbf{x}_k)^\top g_k \mid \mathbf{x}_k\right] = \lVert \nabla f(\mathbf{x}_k) \rVert_2^2$, and the bias–variance split gives $\mathbb{E}\left[\lVert g_k \rVert_2^2 \mid \mathbf{x}_k\right] = \lVert \nabla f(\mathbf{x}_k) \rVert_2^2 + \mathbb{E}\left[\lVert g_k-\nabla f(\mathbf{x}_k)\rVert_2^2\mid \mathbf{x}_k\right] \le \lVert \nabla f(\mathbf{x}_k) \rVert_2^2 + \sigma_B^2$. Therefore

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1}) \mid \mathbf{x}_k\right] \le f(\mathbf{x}_k) - \eta\left(1-\frac{L\eta}{2}\right)\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 + \frac{L\eta^2\sigma_B^2}{2}.
$$

**Step 3 (simplify the coefficient).** Since $\eta \le 1/L$ we have $\frac{L\eta}{2} \le \frac{1}{2}$, so $1-\frac{L\eta}{2} \ge \frac{1}{2}$ and

$$
\mathbb{E}\left[f(\mathbf{x}_{k+1})\mid\mathbf{x}_k\right] \le f(\mathbf{x}_k) - \frac{\eta}{2}\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 + \frac{L\eta^2\sigma_B^2}{2}.
$$

**Step 4 (apply PL).** Strong convexity gives $\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 \ge 2\mu\delta_k$ (Problem L1.5, Step 1). Subtracting $f^\star$ and taking total expectations,

$$
\mathbb{E}[\delta_{k+1}] \le (1-\eta\mu)\,\mathbb{E}[\delta_k] + \frac{L\eta^2\sigma_B^2}{2}.
$$

**Step 5 (unroll the affine recursion).** With $a = 1-\eta\mu \in [0,1)$ and $c = \frac{L\eta^2\sigma_B^2}{2}$, induction gives

$$
\mathbb{E}[\delta_k] \le a^k\delta_0 + c\sum_{j=0}^{k-1}a^j \le a^k\delta_0 + \frac{c}{1-a} = (1-\eta\mu)^k\delta_0 + \frac{L\eta^2\sigma_B^2}{2\eta\mu},
$$

and $\frac{L\eta^2\sigma_B^2}{2\eta\mu} = \frac{\eta L\sigma_B^2}{2\mu}$. $\blacksquare$

$$
\boxed{\mathbb{E}[\delta_k] \le \underbrace{(1-\eta\mu)^k\delta_0}_{\text{geometric transient}} + \underbrace{\frac{\eta L\sigma_B^2}{2\mu}}_{\text{noise floor},\ \sigma_B^2 = \sigma^2/B}}
$$

*Key takeaway:* Two terms, two regimes. Early on the transient dominates and large $\eta$ is best; later the floor dominates and only smaller $\eta$ (or larger $B$) helps. That crossover is precisely why staircase and cosine schedules exist — and why every drop in $\eta$ produces a visible step down in the loss curve.

### Problem L3.4: Explicit versus implicit Euler — where the stability limit comes from

Consider $f(x) = \frac{1}{2}\lambda x^2$, $\lambda \gt 0$. (a) Solve the gradient flow exactly. (b) Show explicit Euler (gradient descent) reproduces the flow only for $\eta\lambda$ small and diverges for $\eta\lambda \gt 2$. (c) Show that implicit Euler — equivalently the proximal-point method — is stable for *every* $\eta \gt 0$, and identify its optimization interpretation.

**Solution.**

**(a) Exact flow.** $\dot{x} = -f'(x) = -\lambda x$ has solution $x(t) = e^{-\lambda t}x_0$: monotone decay, for every $t \gt 0$, with no condition of any kind. Over one time unit $\eta$ the exact multiplier is $m_{\text{exact}} = e^{-\eta\lambda} \in (0,1)$.

**(b) Explicit Euler.** $x_{k+1} = x_k - \eta\lambda x_k = (1-\eta\lambda)x_k$, multiplier $m_{\text{exp}} = 1-\eta\lambda$. This is the two-term truncation of $e^{-\eta\lambda} = 1-\eta\lambda+\frac{(\eta\lambda)^2}{2}-\cdots$, accurate to $O((\eta\lambda)^2)$. But as a *stability* statement it is qualitatively wrong for large $\eta\lambda$: $\lvert m_{\text{exp}} \rvert \lt 1$ only for $\eta\lambda \in (0,2)$, and $\lvert m_{\text{exp}}\rvert \gt 1$ for $\eta\lambda \gt 2$, whereas $m_{\text{exact}} \in (0,1)$ always. In $d$ dimensions the binding mode is $\lambda_{\max}$, giving $\eta \lt 2/\lambda_{\max}$, and the dimensionless product $\eta\lambda_{\max}$ plays the role of a Courant number.

**(c) Implicit Euler.** Evaluate the gradient at the *new* point: $x_{k+1} = x_k - \eta\lambda x_{k+1}$, so

$$
x_{k+1} = \frac{1}{1+\eta\lambda}x_k, \qquad m_{\text{imp}} = \frac{1}{1+\eta\lambda} \in (0,1) \ \text{ for every } \eta \gt 0 .
$$

Unconditionally stable, and monotone (never oscillates). Its optimization meaning: $x_{k+1} = x_k - \eta\nabla f(x_{k+1})$ is exactly the stationarity condition of

$$
x_{k+1} = \arg\min_{z}\left\{ f(z) + \frac{1}{2\eta}\left(z-x_k\right)^2 \right\},
$$

the **proximal-point method**. Indeed, as $\eta \to \infty$, $m_{\text{imp}} \to 0$: the implicit step with an infinite step size jumps straight to the minimizer. The catch is that each step requires solving an optimization problem, which for general $f$ is as hard as the original — hence proximal methods are used when that subproblem is cheap in closed form (e.g. $\ell_1$ soft-thresholding in ISTA, or proximal operators of simple regularizers).

$$
\boxed{m_{\text{exact}} = e^{-\eta\lambda},\quad m_{\text{exp}} = 1-\eta\lambda \ (\text{needs } \eta \lt 2/\lambda),\quad m_{\text{imp}} = \tfrac{1}{1+\eta\lambda} \ (\text{always stable})}
$$

*Key takeaway:* There is no stability limit in gradient *descent* — only in its explicit discretization. Every trick that lets practitioners raise the learning rate (normalization layers, gradient clipping, trust regions, proximal and implicit steps) is, in this language, a move toward a more stable integrator.